# A8 loopback FRA — complete startup and measurement
This notebook contains the entire procedure: **clock setup → helper refresh/FPGA loading → D2S voltage check → fixture connection → DC calibration → 10 Hz–1 MHz sweep → plots and delay fit**. Use this notebook instead of combining cells from previous versions. Run it **cell by cell**, because you must measure two voltages and reconnect the mixer IF between cells.

Copy this notebook and the matching `laser_pll_R8.py`, `laser_pll.py`, `laser_pll_loop_sweep.py`, and `laser_pll_loopback.py` from the current A8 Python ZIP into the same notebook folder. The packaged R8 file is identical to the canonical A8 helper. Your existing bitstream path is used below; its matching `.hwh` must sit alongside it. No new bitstream is needed for this notebook update.

**Before starting:** disconnect the laser controllers from the driven output. For mixer mode, leave **D2S → mixer IF disconnected** until step 4. Initializing clocks/loading the overlay interrupts existing locks. The startup cell ends with the measured DAC command disabled.

| Connection | Hardware channel | Python address |
|---|---|---|
| Measured output in both tests | DAC 229/1 | `rfdc.dac_tiles[1].blocks[1]` |
| Receiver in both tests | ADC 224/3 | `rfdc.adc_tiles[0].blocks[3]` |
| External mixer LO only | TP DAC 228/1 | `rfdc.dac_tiles[0].blocks[1]`, `ol.axi_gpio_12.channel1` |

- **`mode = "mixer"`:** DAC229/1 → D2S → optional BLP → ZAD-1+ IF; TP DAC228/1 → LO; mixer RF → ADC224/3. The intensity ADC224/1 is not used.
- **`mode = "rf"`:** DAC229/1 RF output → suitable RF interface/attenuation → ADC224/3. D2S and external mixer are outside this fixture.

Use the same 90 MHz carrier for comparison. The ZAD-1+ LO is nominally +7 dBm; verify at its LO connector. The **100 mV limit applies to the D2S IF stimulus, not the separate LO**. Record BLP, cabling, termination and attenuation. Mixer loading differs from the original high-impedance controller tee.

## 1. Refresh modules, configure clocks and load the overlay
Leave `initialize_hardware = True` for the first run. To refresh Python in an already initialized session without reprogramming, set it to `False`; an existing A8 `ol` is required. Both choices mute the measured output for this diagnostic. Rerun the subsequent setup/calibration cells after a refresh.


In [ ]:
from pathlib import Path
from datetime import datetime
import importlib
import sys
import json
import time
import re

import pynq
from pynq import Overlay
import xrfdc
import xrfclk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

initialize_hardware = True
custom_lmk_file = Path("/home/xilinx/jupyter_notebooks/LMK_245_76_ExtRef10.txt")
bitfile = "/home/xilinx/Bitfiles/FreqLocksBranch/overlays/TransportPhaseLockFF.bit"

# Refresh the control helper first, then its dependants.
importlib.invalidate_caches()
import laser_pll_R8 as laser_pll
laser_pll = importlib.reload(laser_pll)
# Internal imports of "laser_pll" must resolve to the same R8 module.
sys.modules["laser_pll"] = laser_pll
import laser_pll_loop_sweep as fra
import laser_pll_loopback as lb
fra = importlib.reload(fra)
lb = importlib.reload(lb)
fra.laser_pll = laser_pll
lb.laser_pll = laser_pll
lb.fra = fra

for helper in (laser_pll, fra, lb):
    print(f"{helper.__name__}: {helper.__file__}")
assert getattr(laser_pll, "FPGA_VERSION", None) == 0xA8, "Update laser_pll_R8.py"
assert getattr(fra, "FPGA_VERSION", None) == 0xA8, "Update laser_pll_loop_sweep.py"
assert callable(getattr(fra, "diagnostics", None)), "Sweep helper lacks diagnostics()"
assert getattr(lb, "HELPER_REVISION", 0) >= 2, "Update laser_pll_loopback.py"

if initialize_hardware:
    # Your custom LMK configuration key and LMX frequency are preserved.
    custom_regs = []
    with custom_lmk_file.open() as f:
        for line in f:
            match = re.search(r"(0x[0-9A-F]+)", line, re.IGNORECASE)
            if match:
                custom_regs.append(int(match.group(1), 16))
    if not custom_regs:
        raise ValueError(f"No LMK registers found in {custom_lmk_file}")

    if not xrfclk.xrfclk.lmk_devices or not xrfclk.xrfclk.lmx_devices:
        xrfclk.xrfclk._find_devices()
        xrfclk.xrfclk._read_tics_output()
    lmk_chip = xrfclk.xrfclk.lmk_devices[0]["compatible"]
    xrfclk.xrfclk._Config[lmk_chip][245.761] = custom_regs
    xrfclk.set_ref_clks(lmk_freq=245.761, lmx_freq=491.52)
    time.sleep(2)
    ol = Overlay(bitfile)
    time.sleep(2)
elif "ol" not in globals():
    raise ValueError("No existing ol: use initialize_hardware = True first.")

rfdc = ol.usp_rf_data_converter_0
fpga_version = ol.pll_acquisition_status.read(8) >> 24
print("FPGA version:", hex(fpga_version))
assert fpga_version == 0xA8, "Use the A8 bitstream/HWH"
lb.check_helpers(ol)
lb.stop(ol, mute=True)

adc_tile = rfdc.adc_tiles[0]
dac_tile = rfdc.dac_tiles[0]           # TP LO tile
measured_dac_tile = rfdc.dac_tiles[1]
if initialize_hardware:
    adc_tile.StartUp()
    dac_tile.StartUp()
    measured_dac_tile.StartUp()
    time.sleep(1)
status = rfdc.IPStatus
print("ADC tile 0 PLL locked:", adc_tile.PLLLockStatus)
print("ADC tile 0 state:", status["ADCTileStatus"][0]["TileState"])
print("DAC tile 0 state:", status["DACTileStatus"][0]["TileState"])
print("DAC tile 1 state:", status["DACTileStatus"][1]["TileState"])
print("TileState 15 = startup complete")
print("Startup complete. Measured DAC command is disabled.")


## 2. Select the fixture and configure the TP LO
Start with `mode = "mixer"`. Keep the mixer IF disconnected for the next voltage measurement. The TP LO is set here; check its connector power with the intended LO cable. For direct RF mode the LO is left unchanged, and the D2S voltage-check cell is skipped automatically.

When switching fixtures later, run the final cleanup, move cables with the measured output disabled, then rerun from this step. Keep 90 MHz in both fixtures for a comparable delay measurement. Direct RF can additionally use 800 MHz; the ZAD-1+ cannot.

Edit `fixture_notes` to record whether the BLP is installed, cable lengths, attenuation, termination and measured LO power.


In [ ]:
mode = "mixer"               # "rf" for direct DAC → ADC
carrier_hz = 90e6
fixture_notes = "EDIT: BLP present/absent, cables, loads, attenuation, measured LO dBm"
bias_codes = 0
amplitude_codes = 32
step_codes = 32
output_limits = None
voltage_readings_mV = None
lb.stop(ol, mute=True)
if mode not in ("mixer", "rf"):
    raise ValueError('Choose mode="mixer" or mode="rf".')
if mode == "mixer":
    if not 1e6 <= carrier_hz <= 450e6:
        raise ValueError("Use a mixer-compatible carrier, normally 90 MHz.")
    lo_freq = carrier_hz / 1e6  # RFDC mixer units: MHz
    lo_amp = 0.25              # verify +7 dBm at LO; this fraction is not a power calibration
    dac_block = rfdc.dac_tiles[0].blocks[1]
    mixer_cfg = dict(dac_block.MixerSettings)
    mixer_cfg["Freq"] = lo_freq
    dac_block.MixerSettings = mixer_cfg
    dac_block.UpdateEvent(xrfdc.EVENT_MIXER)
    ol.axi_gpio_12.channel1.write(int(lo_amp * 32767), mask=0xFFFF)


## 3. Measure D2S volts/code — mixer IF disconnected
For mixer mode, disconnect D2S from the IF port and from the controllers. Connect a **DC-coupled, high-impedance** scope or meter to the D2S output. Read its **mean voltage**, not peak-to-peak noise. The cell applies code 0, waits for your first reading, then applies code +100 and waits for the second. Enter signed **mV**: enter `-4` for −4 mV. It ends with the DAC disabled even if you interrupt an input prompt.

These measurements create `output_limits` automatically. No placeholder gain or manual extra setup call is needed. The unloaded reading provides a useful conservative DC bound for a passive load; analog peaking and changed loading still need a scope check. Keep the IF disconnected if either reading fails the voltage check.


In [ ]:
if mode == "mixer":
    lb.check_helpers(ol)
    lb.stop(ol, mute=True)
    ol.pll_iir.write(0, ol.pll_iir.read(0) & ~1)
    dac = rfdc.dac_tiles[1].blocks[1]  # measured DAC 229/1
    cfg = dict(dac.MixerSettings)
    cfg.update(Freq=0.0, PhaseOffset=0.0, EventSource=xrfdc.EVNT_SRC_IMMEDIATE,
               MixerType=xrfdc.MIXER_TYPE_FINE, MixerMode=xrfdc.MIXER_MODE_C2R,
               FineMixerScale=xrfdc.MIXER_SCALE_1P0)
    dac.MixerSettings = cfg
    dac.ResetNCOPhase()
    dac.UpdateEvent(xrfdc.EVENT_MIXER)
    try:
        laser_pll.set_dc_output(ol, 0)
        laser_pll.enable(ol)
        lb._wait_dc(ol)
        time.sleep(0.2)
        v0_mV = float(input("Mean D2S voltage at code 0, in mV: "))
        laser_pll.set_dc_output(ol, 100)
        time.sleep(0.2)
        v100_mV = float(input("Mean D2S voltage at code +100, in mV: "))
    finally:
        laser_pll.enable(ol, False)
        laser_pll.set_dc_output(ol, 0)

    voltage_readings_mV = {"code_0": v0_mV, "code_100": v100_mV}
    output_limits = lb.voltage_limits(v0_mV * 1e-3, v100_mV * 1e-3,
                                     maximum_abs_V=0.100, margin_V=0.020)
    # Initial probe/calibration excursions: <=5 mV, also <=32 DAC codes.
    amplitude_codes = min(32, int(0.005 / abs(output_limits["volts_per_code"])))
    step_codes = amplitude_codes
    if amplitude_codes < 1:
        raise ValueError("Unexpectedly large volts/code; check the measured voltage units.")
    peak_V = lb.check_output(bias_codes, max(amplitude_codes, step_codes), output_limits)
    print("D2S model:", output_limits)
    print(f"Predicted maximum |V|: {peak_V*1e3:.2f} mV, plus 20 mV headroom")
    print(f"Probe: {amplitude_codes} codes peak; calibration: +/-{step_codes} codes")
    print("Voltage check passed. DAC disabled; connect the mixer IF for step 4.")
else:
    print("Direct RF mode: no D2S voltage check required.")


## 4. Connect the fixture, then enable the bounded stimulus
**Mixer:** after the voltage check passes, connect D2S → ZAD-1+ **IF**, TP DAC228/1 → **LO**, and mixer **RF** → ADC224/3. **Direct RF:** connect DAC229/1 RF → ADC224/3 through your RF interface/attenuation.

Keep a high-impedance scope on D2S for the first mixer measurement and verify actual peaks. The helper checks `|offset + slope*bias| + |slope|*amplitude + 20 mV < 100 mV` for setup, both calibration levels, and every sine point. This is a measured-model software bound, not a hardware clamp. Changed external gain, offset or loading requires repeating the voltage check.

The next cell automatically passes the `output_limits` produced in step 3. **Do not substitute an older setup cell.** It preserves the ADC converter settings, programs the measured DAC and receive NCO, and waits for DC mode to settle.


In [ ]:
setup_info = lb.setup(ol, mode=mode, carrier_hz=carrier_hz,
                      bias_codes=bias_codes, output_limits=output_limits)
print(lb.receiver_settings(ol))


## 5. Calibrate the I/Q receiver and measure one 1 kHz point
This is separate from the D2S meter calibration. A low–high–low sequence at `bias_codes ± step_codes` measures the real I/Q response vector in ADC counts per DAC code. Subtraction removes fixed LO leakage and receiver offsets. Normalizing to this vector gives a DC-normalized transfer function, including physical polarity.

The first sine point runs for about one second, then the probe ramps off. Confirm D2S voltage stays below 100 mV in magnitude. If calibration drifts or vanishes, inspect LO level, cabling, DC coupling and matched carrier frequencies. Increase integration if noise is limiting; do not return to the old 512-code calibration step without checking its voltage bound. This measurement uses I/Q amplitude response, not CORDIC or the PLL discriminator.


In [ ]:
calibration = lb.calibrate(ol, step_codes=step_codes, integration_s=1.0)
print({k: calibration[k] for k in ('gain_i', 'gain_q', 'drift_fraction')})
point = lb.measure_point(ol, 1e3, amplitude_codes=amplitude_codes, minimum_measure_s=1.0)
h, perpendicular = lb.project(point, calibration)
print('Actual Hz:', point['frequency_hz'], 'H:', h, 'orthogonal response:', perpendicular)
print('Diagnostics:', point['diagnostics'])


## 6. Sweep 10 Hz–1 MHz and save data
Reference multiplication happens at 245.76 MHz **before** coherent integration. One-second blocks have approximately 0.5 Hz one-sided rectangular-window noise bandwidth per demodulated quadrature. Eight complex repeats reduce independent noise further; adaptive repeats extend to 24. Polarity alternates to reduce fixed pickup bias. Drift and distortion do not disappear through averaging.

The 61-point sweep takes roughly 9–27 minutes, plus software overhead. CSV files are updated after each block/point in **`loopback_data/` beside this notebook**. The cell prints absolute paths. JSON includes the voltage readings, output limit, carrier, gain vector, converter settings and fixture notes. This saves transfer-function results, not PSD or raw DMA samples.

If interrupted, completed blocks remain on disk; each probe's helper cleanup ramps its sine to zero. Run the final cleanup cell to disable the measured output before recabling.


In [ ]:
output_dir = Path('loopback_data')
output_dir.mkdir(exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
csv_path = output_dir / f'loopback_{mode}_{stamp}.csv'
metadata = dict(voltage_readings_mV=voltage_readings_mV, output_limits=output_limits, helper_revision=lb.HELPER_REVISION,
                calibration_step_codes=step_codes, mode=mode, carrier_hz=carrier_hz, bias_codes=bias_codes,
                fpga_version=hex(ol.pll_acquisition_status.read(8) >> 24),
                fabric_sample_rate_hz=lb.FS_HZ, iq_units='post-mixer ADC counts',
                amplitude_codes=amplitude_codes, gain_i=calibration['gain_i'],
                gain_q=calibration['gain_q'], drift_fraction=calibration['drift_fraction'],
                settings=lb.receiver_settings(ol),
                fixture_notes=fixture_notes)
csv_path.with_suffix('.json').write_text(json.dumps(metadata, indent=2))
frequencies_hz = np.geomspace(10, 1e6, 61)
rows = lb.sweep(ol, frequencies_hz, calibration,
                amplitude_codes=amplitude_codes, minimum_measure_s=1.0,
                repeats=8, max_repeats=24, target_snr=30, csv_path=csv_path)
print('Saved:', csv_path.resolve())
print('Metadata:', csv_path.with_suffix('.json').resolve())
print('Raw complex blocks:', csv_path.with_name(csv_path.stem + '_blocks.csv').resolve())


## 7. Plot magnitude/phase and estimate delay
Magnitude is normalized to the DC response. A pure delay gives a negative phase slope: `phase = constant − 2π f × delay`. The fit uses reliable points from 10 kHz to 1 MHz and includes converters, external fixture and digital mixer/FIR. It does not separate their individual delays. Phase curvature means the fitted delay is band dependent. The PNG is saved alongside the CSV.


In [ ]:
frequency_hz = np.array([r['frequency_hz'] for r in rows])
h = np.array([complex(r['h_real'], r['h_imag']) for r in rows])
sem = np.array([r['complex_sem'] for r in rows])
reliable = np.array([r['reliable'] for r in rows])
phase_deg = np.rad2deg(np.unwrap(np.angle(h)))
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 7))
axes[0].errorbar(frequency_hz, 20*np.log10(abs(h)),
                 yerr=20/np.log(10)*sem/abs(h), fmt='.-', label=mode)
axes[1].errorbar(frequency_hz, phase_deg, yerr=np.rad2deg(sem/abs(h)), fmt='.-')
axes[0].plot(frequency_hz[~reliable], 20*np.log10(abs(h[~reliable])), 'rx', label='low SNR')
axes[0].set_ylabel('DC-normalized magnitude (dB)')
axes[1].set_ylabel('Unwrapped phase (deg)')
axes[1].set_xlabel('Modulation frequency (Hz)')
for ax in axes:
    ax.set_xscale('log'); ax.grid(True, which='both', alpha=.25)
y_low, y_high = axes[0].get_ylim()
if y_high-y_low < 1.0:
    middle = (y_low+y_high)/2
    axes[0].set_ylim(middle-.5, middle+.5)
axes[0].legend()
fig.suptitle(f'{mode} loopback, {carrier_hz/1e6:g} MHz carrier')
fig.tight_layout()
fig.savefig(csv_path.with_suffix('.png'), dpi=180)
print('Plot:', csv_path.with_suffix('.png').resolve())
delay = None
try:
    delay = lb.fit_delay(rows, minimum_hz=1e4, maximum_hz=1e6)
except ValueError as exc:
    print('Delay fit unavailable:', exc)
else:
    print(f"Fitted total delay: {delay['delay_s']*1e9:.1f} ns")
    print(f"Phase-line residual RMS: {np.rad2deg(delay['phase_residual_rms_rad']):.2f} deg")
plt.show()


## 8. Interpret, repeat, and clean up
1. Repeat selected points at half amplitude. Their **normalized complex responses** should agree within the uncertainty; otherwise suspect compression, rail events or poor SNR.
2. Repeat the DC calibration after the sweep. A changed I/Q direction indicates carrier phase drift; a changed magnitude indicates gain drift. Keep converter/NCO state fixed within a sweep.
3. Measure both modes at the same 90 MHz carrier. Divide the mixer-path complex response by the direct response, then unwrap phase. This removes much of the shared digital/converter contribution; different external RF cables, the mixer, and DAC analog response near DC versus RF still contribute.
4. An extra known cable should add its propagation delay (roughly 5 ns/m for many coax types; use that cable's velocity factor). Repeat after RFDC resets to check startup-dependent latency.
5. For a dispersive LPF, the phase curve bends: a single delay fit is only a band-average slope. Inspect the curve and residual; do not equate group delay at one frequency with phase delay everywhere.

The measurement includes DAC packing/interpolation, DAC/ADC conversion and the existing mixer/FIR. It excludes CORDIC, phase unwrapping, phase scaling, analog controllers and laser dynamics. No software-trigger timing enters the phase estimate. Absolute delay cannot distinguish DAC from ADC latency.

Run this cleanup cell before changing wiring. It mutes the measured DAC; it leaves the TP LO and second PLL as configured.

To measure the other fixture: run cleanup below, move cables, change `mode` in step 2, then rerun steps 2–8. Keep the startup cell alone unless you intend to reinitialize hardware. For helper-only refresh use `initialize_hardware = False` and then redo setup/calibration. To stop a sweep, interrupt the running cell and execute cleanup; do not change wiring while it is still running.


In [ ]:
lb.stop(ol, mute=True)
# To resume a laser lock, run the ordinary PLL configure/acquisition cells.
# laser_pll.configure() restores the measured DAC NCO to zero frequency/phase.
